# 🚀 Huấn Luyện Phân Loại Cảm Xúc Tiếng Việt - Google Colab

Notebook này giúp huấn luyện dự án PhanLoaiCamXuc trên GPU Colab.

---

## ⚠️ BƯỚC QUAN TRỌNG ĐẦU TIÊN: Kích hoạt GPU trước khi chạy!

1. Chọn **Runtime** → **Change runtime type**
2. Chọn **T4 GPU** hoặc **A100 GPU**
3. Nhấp **Save**

Sau đó mới bắt đầu chạy từng cell bên dưới.

## Bước 1: Mount Google Drive

Lưu dự án vào Google Drive để không bị mất khi Colab tắt.

In [ ]:
from google.colab import drive
import os

print('📂 Đang mount Google Drive...')
drive.mount('/content/drive')

# Đường dẫn lưu dự án trên Drive
PROJECT_DIR = '/content/drive/MyDrive/PhanLoaiCamXuc'
print(f'✅ Đã mount Drive! Dự án sẽ được lưu tại: {PROJECT_DIR}')

## Bước 2: Clone hoặc Pull Repository từ GitHub

- Nếu chưa có repo: Clone từ GitHub
- Nếu đã có repo: Pull cập nhật mới nhất
- Không xóa folder cũ để giữ data và model đã train

In [ ]:
import os
import subprocess

GITHUB_REPO = 'https://github.com/Escanor292/PhanLoaiCamXuc.git'
PROJECT_DIR = '/content/drive/MyDrive/PhanLoaiCamXuc'

if os.path.exists(PROJECT_DIR):
    print('📂 Đã tìm thấy thư mục dự án trên Drive.')
    print('🔄 Đang pull cập nhật mới nhất...')
    %cd {PROJECT_DIR}
    !git pull
else:
    print('📂 Chưa có thư mục dự án.')
    print('🔄 Đang clone từ GitHub...')
    !git clone {GITHUB_REPO} {PROJECT_DIR}
    %cd {PROJECT_DIR}

print(f'✅ Sẵn sàng! Đang làm việc tại: {os.getcwd()}')

## Bước 3: Cài đặt Dependencies

Cài đặt các thư viện cần thiết từ requirements.txt và thêm thư viện bổ sung.

In [ ]:
print('📦 Đang cài đặt dependencies...')

# Cài đặt từ requirements.txt (nếu có)
if os.path.exists('ND/requirements.txt'):
    !pip install -q -r ND/requirements.txt
elif os.path.exists('requirements.txt'):
    !pip install -q -r requirements.txt
else:
    print('⚠️ Không tìm thấy requirements.txt, cài đặt thủ công...')
    !pip install -q torch transformers pandas numpy scikit-learn matplotlib tqdm

# Cài đặt thêm thư viện cần thiết cho Colab
!pip install -q huggingface_hub accelerate sentencepiece python-dotenv

print('✅ Cài đặt hoàn tất!')

## Bước 4: Kiểm tra GPU

Xác nhận GPU đã được kích hoạt trước khi huấn luyện.

In [ ]:
import torch

print('=' * 60)
print('THÔNG TIN GPU')
print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'GPU Available   : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device      : {gpu_name}')
    print(f'GPU Memory      : {gpu_mem:.1f} GB')
    print('\n🎉 GPU đã sẵn sàng! Có thể bắt đầu huấn luyện.')
else:
    print('\n⚠️ CHƯA CÓ GPU!')
    print('Vui lòng vào Runtime → Change runtime type → T4 GPU rồi chạy lại.')

## Bước 5: Đăng nhập Hugging Face

Cần đăng nhập để tải PhoBERT model từ Hugging Face Hub.

**Lưu ý:** Không hard-code token. Sẽ hiện popup để nhập token.

In [ ]:
from huggingface_hub import notebook_login

print('🔐 Đang đăng nhập Hugging Face...')
print('Vui lòng nhập Hugging Face token khi popup hiện ra.')
print('(Nếu chưa có token, lấy tại: https://huggingface.co/settings/tokens)')
notebook_login()
print('✅ Đăng nhập thành công!')

## Bước 6: Kiểm tra Cấu trúc Thư Mục Quan Trọng

Kiểm tra các thư mục: data/, model_registry/, saved_model/

⚠️ **CẢNH BÁO QUAN TRỌNG:**
- KHÔNG xóa thư mục `model_registry/`
- KHÔNG xóa file `model_registry/data_tracker.json`
- KHÔNG xóa thư mục `data/`
- KHÔNG xóa thư mục `saved_model/`

Các thư mục này chứa:
- `data_tracker.json`: Theo dõi dữ liệu đã train (incremental training)
- `registry.json`: Danh sách model đã đăng ký
- `models/`: Các model checkpoint đã train
- `saved_model/`: Model production đang sử dụng

In [ ]:
import os

print('=' * 60)
print('KIỂM TRA CẤU TRÚC THƯ MỤC')
print('=' * 60)

for dir_name in ['data', 'model_registry', 'saved_model']:
    path = dir_name
    if os.path.exists(path):
        print(f'✅ {dir_name}/ - TỒN TẠI')
        # List nội dung
        items = os.listdir(path)
        if items:
            print(f'   Nội dung: {len(items)} items')
            for item in items[:5]:  # Hiện 5 items đầu
                item_path = os.path.join(path, item)
                if os.path.isdir(item_path):
                    print(f'   📁 {item}/')
                else:
                    size = os.path.getsize(item_path) / 1024
                    print(f'   📄 {item} ({size:.1f} KB)')
            if len(items) > 5:
                print(f'   ... và {len(items) - 5} items khác')
    else:
        print(f'⚠️  {dir_name}/ - CHƯA TỒN TẠI (sẽ được tạo tự động)')
    print()

# Cảnh báo đặc biệt về data_tracker.json
tracker_file = 'model_registry/data_tracker.json'
if os.path.exists(tracker_file):
    print('⚠️  CẢNH BÁO: model_registry/data_tracker.json ĐÃ TỒN TẠI')
    print('   File này theo dõi dữ liệu đã train. KHÔNG XÓA file này!')
    print('   Nếu xóa, hệ thống sẽ train lại toàn bộ dữ liệu từ đầu.')
else:
    print('ℹ️  model_registry/data_tracker.json chưa tồn tại (sẽ được tạo khi train lần đầu)')

print('=' * 60)

## Bước 7: Liệt kê File Dữ Liệu (data/member_*.csv)

Hệ thống tự động quét các file CSV trong thư mục data/ để huấn luyện.
- Bỏ qua file có chứa "TEMPLATE"
- Bỏ qua file "merged_temp.csv" (file tạm thời)

In [ ]:
import glob
import os

print('=' * 60)
print('FILE DỮ LIỆU TRONG data/')
print('=' * 60)

csv_files = glob.glob('data/*.csv')
# Lọc bỏ TEMPLATE và merged_temp
csv_files = [f for f in csv_files 
             if 'TEMPLATE' not in f.upper() 
             and 'merged_temp' not in f.lower()
             and '.gitkeep' not in f]
csv_files = sorted(csv_files)

if csv_files:
    print(f'Tìm thấy {len(csv_files)} file dữ liệu:\n')
    total_size = 0
    for i, f in enumerate(csv_files, 1):
        size = os.path.getsize(f) / 1024
        total_size += size
        print(f'{i}. 📄 {os.path.basename(f):30s} ({size:.0f} KB)')
    print(f'\nTổng kích thước: {total_size:.0f} KB ({total_size/1024:.1f} MB)')
else:
    print('⚠️ Không tìm thấy file dữ liệu nào!')
    print('Vui lòng thêm file CSV vào thư mục data/ trước khi huấn luyện.')

print('=' * 60)

## Bước 8: Kiểm tra Danh sách Nhãn Cảm Xúc từ config.py

Dự án sử dụng 16 nhãn cảm xúc. Không thay đổi danh sách này nếu không được yêu cầu.

In [ ]:
import sys
sys.path.append(os.getcwd())

from config import Config

print('=' * 60)
print('DANH SÁCH NHÃN CẢM XÚC')
print('=' * 60)
print(f'Tổng số nhãn: {Config.NUM_LABELS}\n')

print('Nhãn tiếng Anh - Tiếng Việt:')
for i, (label_en, label_vi) in enumerate(Config.EMOTION_LABELS_VI.items(), 1):
    print(f'{i:2d}. {label_en:15s} - {label_vi}')

print('\n' + '=' * 60)
print('⚠️ KHÔNG THAY ĐỔI danh sách nhãn nếu không được yêu cầu.')
print('=' * 60)

## Bước 9: Kiểm tra Dữ Liệu Mới với Data Tracker

Chạy `python data_tracker.py check` để xem:
- Tổng số mẫu trong các file CSV
- Số mẫu mới chưa từng train
- Số mẫu đã train rồi (sẽ bị bỏ qua)

Đây là cơ chế **incremental training** - chỉ train dữ liệu mới!

## Bước 9.5: Tải Model Tốt Nhất từ Hugging Face (Tùy chọn - Khuyên dùng)

Trước khi huấn luyện, tải model tốt nhất từ Hugging Face về local để enable transfer learning.

**Lợi ích:**
- Model mới sẽ học từ kiến thức của model tốt nhất hiện có
- Giảm thời gian training
- Cải thiện độ chính xác

**Cách thức:**
1. Kiểm tra model tốt nhất trong local registry
2. Nếu chưa có local, tải từ Hugging Face Hub
3. Lưu vào `model_registry/models/` để train_simple.py sử dụng

In [ ]:
import sys
sys.path.append(os.getcwd())

from model_registry import ModelRegistry
from model_sharing import ModelSharing
import os

print('=' * 60)
print('TẢI MODEL TỐT NHẤT TỪ HUGGING FACE')
print('=' * 60)
print()

# Khởi tạo registry
registry = ModelRegistry()
best_model = registry.get_best_model()

if best_model:
    model_id = best_model['model_id']
    local_path = best_model['path']
    
    print(f'🏆 Model tốt nhất trong registry: {model_id}')
    print(f'📂 Local path: {local_path}')
    
    # Kiểm tra xem model đã tồn tại local chưa
    if os.path.exists(local_path) and os.listdir(local_path):
        print(f'✅ Model đã tồn tại local. Không cần tải lại.')
    else:
        print(f'⚠️  Model chưa tồn tại local hoặc folder trống.')
        print(f'📥 Đang tải từ Hugging Face Hub...')
        
        try:
            sharing = ModelSharing()
            # Download model về local
            success = sharing.download_model(model_id, target_path=local_path)
            
            if success:
                print(f'✅ Đã tải model {model_id} thành công!')
                print(f'📂 Đã lưu tại: {local_path}')
            else:
                print(f'⚠️  Tải model thất bại. Sẽ train từ local nếu có, hoặc train từ đầu.')
        except Exception as e:
            print(f'❌ Lỗi khi tải model: {e}')
            print(f'   Sẽ tiếp tục với model local hoặc train từ đầu.')
else:
    print('ℹ️  Chưa có model nào trong registry.')
    print('   Hệ thống sẽ train từ đầu (không có transfer learning).')

print()
print('=' * 60)
print('⚠️  KHÔNG reset data_tracker, KHÔNG xóa data/model_registry/saved_model')
print('=' * 60)

## Bước 10: Huấn Luyện Model 🚀

Chạy lệnh training chính: `python train_simple.py`

**Logic training được giữ nguyên:**
1. DataTracker quét data/member_*.csv
2. Lọc chỉ lấy dữ liệu MỚI chưa train
3. Gộp vào merged_temp.csv
4. Train với Hybrid PhoBERT + BiLSTM + Self-Attention
5. Transfer learning từ model tốt nhất (nếu có)
6. Đăng ký model vào model_registry/
7. Cập nhật data_tracker.json (đánh dấu dữ liệu đã train)

**Cài đặt mặc định:**
- Model: Hybrid PhoBERT (mạnh nhất cho tiếng Việt)
- Epochs: 5 (tối ưu cho Hybrid)
- Learning Rate: 2e-5
- Batch Size: 16
- LSTM Hidden Size: 256

In [ ]:
print('=' * 60)
print('BẮT ĐẦU HUẤN LUYỆN')
print('=' * 60)
print()
print('Lệnh: python train_simple.py')
print('Logic: data/member_*.csv → DataTracker → merged_temp.csv → train_simple.py → model_registry')
print()
print('⏱️  Quá trình này có thể tốn 10-30 phút tùy vào lượng dữ liệu mới.')
print('🔥 Vui lòng kiên nhẫn đợi...')
print('=' * 60)
print()

!python train_simple.py

## Bước 11: Hiển thị Kết Quả Training (results.txt)

Tìm và hiển thị file results.txt mới nhất nếu có.

In [ ]:
import glob
import os

print('=' * 60)
print('KẾT QUẢ TRAINING')
print('=' * 60)

# Tìm results.txt trong các thư mục experiments/
result_files = glob.glob('experiments/*/results.txt', recursive=True)

if result_files:
    # Lấy file mới nhất
    latest_result = max(result_files, key=os.path.getmtime)
    print(f'\n📄 File kết quả mới nhất: {latest_result}\n')
    print('-' * 60)
    with open(latest_result, 'r', encoding='utf-8') as f:
        print(f.read())
    print('-' * 60)
else:
    print('\n⚠️ Không tìm thấy file results.txt.')
    print('Kết quả có thể được hiển thị trực tiếp trong output training bên trên.')

print('=' * 60)

## Bước 12: Xem Danh sách Model Đã Đăng Ký

Chạy `python model_registry.py list` để xem:
- Các model đã train
- Metrics (Macro F1, Micro F1, Test Loss)
- Model nào đang là BEST
- Model nào đang là PRODUCTION

In [ ]:
print('=' * 60)
print('DANH SÁCH MODEL ĐÃ ĐĂNG KÝ')
print('=' * 60)
print()
!python model_registry.py list

## Bước 13: Test Dự Đoán (Tùy chọn)

Thử dự đoán cảm xúc với một câu bất kỳ.

In [ ]:
# Thay đổi câu test ở đây
test_text = "Sản phẩm này rất tốt, giao hàng nhanh và đóng gói cẩn thận!"

print(f'📝 Câu test: "{test_text}"\n')
!python predict.py --text "{test_text}"

## Bước 14: Git Commit và Push (Tùy chọn - COMMENT MẶC ĐỊNH)

⚠️ **KHÔNG TỰ ĐỘNG PUSH** nếu bạn chưa xác nhận!

Nếu muốn lưu kết quả training lên GitHub:
1. Bỏ comment (xóa #) ở các lệnh git bên dưới
2. Điền message commit của bạn
3. Chạy cell này

**Lưu ý:**
- KHÔNG commit model quá lớn (repo không muốn lưu model)
- Ưu tiên giữ model qua Hugging Face Hub
- Chỉ commit metadata: model_registry/registry.json, model_registry/data_tracker.json

In [ ]:
# === BẮT ĐẦU: BỎ COMMENT ĐỂ DÙNG ===

# Cấu hình git
# !git config --global user.email "your-email@example.com"
# !git config --global user.name "Your Name"

# Kiểm tra trạng thái
# !git status

# Add các file metadata (KHÔNG add model nặng)
# !git add model_registry/registry.json
# !git add model_registry/data_tracker.json
# !git add experiments/  # Thêm các biểu đồ kết quả

# Commit
# COMMIT_MESSAGE = "Cập nhật kết quả training trên Colab"
# !git commit -m "{COMMIT_MESSAGE}"

# Push lên GitHub
# !git push

# === KẾT THÚC ===

print('ℹ️  Cell git commit/push đang được comment mặc định.')
print('   Bỏ comment và điền thông tin của bạn nếu muốn push lên GitHub.')

## Bước 15: Tóm Tắt và Hướng Dẫn Tiếp Theo

### ✅ Đã hoàn thành:
1. ✅ Mount Google Drive
2. ✅ Clone/Pull repository từ GitHub
3. ✅ Cài đặt dependencies
4. ✅ Kiểm tra GPU
5. ✅ Đăng nhập Hugging Face
6. ✅ Kiểm tra cấu trúc thư mục (data/, model_registry/, saved_model/)
7. ✅ Liệt kê file dữ liệu (data/member_*.csv)
8. ✅ Kiểm tra danh sách nhãn cảm xúc
9. ✅ Kiểm tra dữ liệu mới với Data Tracker
10. ✅ Chạy training với train_simple.py
11. ✅ Hiển thị kết quả training
12. ✅ Xem danh sách model đã đăng ký

### 🎯 Các bước tiếp theo:
1. **Xem chi tiết model:** `python model_registry.py info --model-id <model_id>`
2. **So sánh model:** `python compare_experiments.py`
3. **Chạy thử interactive:** `python demo_phobert.py --mode interactive`
4. **Đẩy model lên Hugging Face:** (nếu cấu hình AUTO_SYNC_CLOUD=True)

### ⚠️ Nguyên tắc an toàn (ĐÃ TUÂN THỦ):
- ✅ KHÔNG dùng rm -rf với data/, model_registry/, saved_model/
- ✅ KHÔNG sửa train_simple.py, train_with_args.py, config.py
- ✅ KHÔNG hard-code GitHub token hoặc Hugging Face token
- ✅ KHÔNG tự động reset data tracker
- ✅ KHÔNG xóa dữ liệu hoặc model
- ✅ Logic training được giữ nguyên 100%

### 📝 Lưu ý quan trọng:
- Dự án được lưu trên Google Drive tại: `/content/drive/MyDrive/PhanLoaiCamXuc`
- Khi chạy lại lần sau, chỉ cần mount Drive và pull cập nhật mới nhất
- Data tracker tự động theo dõi dữ liệu đã train, không train lại dữ liệu cũ
- Model tốt nhất được tự động lưu trong model_registry/

### 🔄 Để chạy lại lần sau:
1. Mở notebook này trên Colab
2. Runtime → Change runtime type → GPU
3. Chạy từ Bước 1 đến Bước 12
- Hệ thống sẽ tự động pull cập nhật và train chỉ dữ liệu mới!